# 03 — Modeling

Three very different approaches to the same problem. I'll build them in order of complexity — not because the most complex is best, but because going simple-to-complex shows exactly where each model breaks down and why.

The lineup:
1. **ARIMA(5,1,0)** — classical time series, univariate
2. **LSTM** — deep learning, sequential
3. **Linear Regression** — simple multivariate baseline (added after the fact, because "does a straight line beat a neural network?" is a question worth asking)
4. **XGBoost** — gradient-boosted trees, the clear winner


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns, joblib
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
from xgboost import XGBRegressor
from statsmodels.tsa.arima.model import ARIMA
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.preprocessing.sequence import TimeseriesGenerator

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
ROOT = Path("..")

df = pd.read_csv(ROOT / "data" / "processed" / "engineered_dataset.csv")
df["Previous_Lag"] = df["Previous_Price"].shift(1).bfill()
df.dropna(inplace=True)

X = df.drop(columns=["Food_Price"])
y = df["Food_Price"]
split = int(len(df) * 0.8)

print(f"Dataset: {df.shape}  |  Train: {split} rows  |  Test: {len(df)-split} rows")


## Model 1 — ARIMA(5,1,0)

**Why ARIMA at all?** It's the canonical baseline for time series regression. Before throwing deep learning or boosting at a problem, you want to know what the classical approach can do.

**Order selection:** ARIMA(p=5, d=1, q=0).
- `d=1`: one round of differencing to make the series stationary (verified with ADF test during development)
- `p=5`: 5 autoregressive lags — captures up to 5 months of "memory" in the price series
- `q=0`: no moving-average component; the series doesn't show significant MA structure in its ACF

**The fundamental limitation:** ARIMA is univariate. It sees only the price series — not Temperature, not Demand_Index, not any of the 16 features we carefully designed. It's predicting tomorrow's price using only the last 5 days' prices. That's the floor.


In [ ]:
# ARIMA works on monthly aggregated series
ts = df.groupby(["Year","Month"])["Food_Price"].mean().reset_index()
ts["date"] = pd.to_datetime(ts[["Year","Month"]].assign(DAY=1))
ts = ts.sort_values("date").set_index("date")

print(f"Time series length: {len(ts)} monthly observations")
print(f"Date range: {ts.index.min().date()} → {ts.index.max().date()}")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(ts.index, ts["Food_Price"], lw=2, color="#4C72B0")
ax.set_title("Monthly average food price — the series ARIMA actually sees", fontsize=11)
ax.set_xlabel("Date"); ax.set_ylabel("Mean Food Price (USD)")
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/03_monthly_series.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# Load saved ARIMA and evaluate
arima_model = joblib.load(ROOT / "outputs/models/arima_model.pkl")
arima_preds = arima_model.predict(start=0, end=len(ts)-1).values
actual_ts   = ts["Food_Price"].values

arima_rmse = np.sqrt(mean_squared_error(actual_ts, arima_preds))
arima_r2   = r2_score(actual_ts, arima_preds)
print(f"ARIMA  RMSE: {arima_rmse:.4f}  |  R²: {arima_r2:.4f}")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ts.index, actual_ts, lw=2, color="#4C72B0", label="Actual")
ax.plot(ts.index, arima_preds, lw=2, color="#DD4444", ls="--", label="ARIMA predictions")
ax.set_title(f"ARIMA predicts near-mean values and misses all real variance — R²={arima_r2:.3f}", fontsize=11)
ax.set_xlabel("Date"); ax.set_ylabel("Food Price (USD)"); ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/03_arima_predictions.png", dpi=150, bbox_inches="tight")
plt.show()


**R² = -0.35. That's worse than predicting the mean for every observation.** The ARIMA line barely moves — it hovers around \$5.50 while the actual series swings between \$4 and \$8. This isn't surprising in retrospect: the price series has no strong autocorrelation once you account for the fact that prices are driven by external variables (supply shocks, demand surges), not by their own history alone. ARIMA without covariates was never going to work here.


## Model 2 — LSTM (window=12)

**Why LSTM?** Food prices have temporal dependencies — last month's drought affects this month's supply. LSTM's gating mechanism (input, forget, output gates) is designed to selectively retain long-range dependencies. The hypothesis was that it would learn the seasonal cycles and carry relevant history forward.

**Architecture:**
- LSTM(64, return_sequences=True) → captures long-range patterns
- Dropout(0.2) → regularisation; prevents memorising the training curve
- LSTM(32) → second-pass compression
- Dense(1) → single price output

**Window size = 12:** One full year of lookback. Each prediction uses the 12 preceding monthly averages as context.


In [ ]:
series = ts["Food_Price"].values.reshape(-1, 1)
scaler = joblib.load(ROOT / "outputs/models/lstm_scaler.pkl")
scaled = scaler.transform(series)
window = 12

generator    = TimeseriesGenerator(scaled, scaled, length=window, batch_size=1)
lstm_model   = load_model(str(ROOT / "outputs/models/lstm_model.h5"), compile=False)
preds_scaled = lstm_model.predict(generator, verbose=0)
lstm_preds   = scaler.inverse_transform(preds_scaled).flatten()
actual_lstm  = series[window:].flatten()

lstm_rmse = np.sqrt(mean_squared_error(actual_lstm, lstm_preds))
lstm_r2   = r2_score(actual_lstm, lstm_preds)
print(f"LSTM  RMSE: {lstm_rmse:.4f}  |  R²: {lstm_r2:.4f}")

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(ts.index[window:], actual_lstm, lw=2, color="#4C72B0", label="Actual")
ax.plot(ts.index[window:], lstm_preds, lw=2, color="#E8994D", ls="--", label="LSTM predictions")
ax.set_title(f"LSTM predictions are nearly flat (~$5.07) regardless of actual price — R²={lstm_r2:.3f}", fontsize=11)
ax.set_xlabel("Date"); ax.set_ylabel("Food Price (USD)"); ax.legend()
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/03_lstm_predictions.png", dpi=150, bbox_inches="tight")
plt.show()


**R² = -0.09.** The LSTM predictions barely vary — they cluster around \$5.07 while the actual price swings by \$4. It outperforms ARIMA slightly but still fails the basic test of "does it follow the ups and downs?"

Why? Two reasons I traced after the fact:
1. **We fed LSTM only the univariate price series** — the same information ARIMA had. All 16 features were discarded by the monthly aggregation step.
2. **168 training observations (months) is genuinely small for an LSTM.** These models typically need thousands of timesteps to learn meaningful gating patterns. With 168 points and EarlyStopping kicking in at epoch ~12, the network never really trains.

A proper multivariate LSTM using all daily observations would likely perform differently. That's the next experiment if we revisit this.


## Model 3 — Linear Regression (Baseline)

Before XGBoost, I added a linear regression using all 27 engineered features. The question: if Previous_Price has r=0.991 with the target, how good is a straight-line model?


In [ ]:
categorical = ["Season"]
numerical   = X.drop(columns=categorical).columns.tolist()
preprocessor = ColumnTransformer([
    ("cat",   OneHotEncoder(drop="first"), categorical),
    ("scale", StandardScaler(),            numerical),
])

lr_pipe = Pipeline([("prep", preprocessor), ("model", LinearRegression())])

X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]

lr_pipe.fit(X_train, y_train)
lr_preds = lr_pipe.predict(X_test)

lr_rmse = np.sqrt(mean_squared_error(y_test, lr_preds))
lr_r2   = r2_score(y_test, lr_preds)
lr_mape = mean_absolute_percentage_error(y_test, lr_preds) * 100
print(f"Linear Regression  RMSE: {lr_rmse:.4f}  |  R²: {lr_r2:.4f}  |  MAPE: {lr_mape:.2f}%")


R² = 0.98 with a straight line. That sets a high bar for XGBoost. A model that doesn't beat 0.98 R² isn't adding value over a linear baseline — and that's a useful sanity check many ML projects skip.


## Model 4 — XGBoost

**Why XGBoost over Random Forest?** Both are ensemble tree methods, but XGBoost:
- Builds trees sequentially (each corrects the previous one's errors) rather than independently
- Has built-in L1/L2 regularisation
- Handles the small-dataset situation better with gamma and colsample_bytree controls

**Hyperparameters (and why each one):**

| Parameter | Value | Reason |
|---|---|---|
| max_depth | 3 | Shallow trees — prevents memorising 1,000 rows |
| colsample_bytree | 0.5 | Uses 50% of features per tree — forces generalisation |
| reg_lambda | 1 | L2 regularisation on leaf weights |
| gamma | 0.2 | Minimum loss reduction required to make a split — prunes useless splits |
| random_state | 42 | Reproducibility |

**Validation:** TimeSeriesSplit with 5 folds — respects temporal ordering.


In [ ]:
xgb_preprocessor = ColumnTransformer([
    ("cat",   OneHotEncoder(drop="first"), categorical),
    ("scale", StandardScaler(),            numerical),
])

xgb_pipe = Pipeline([
    ("preprocessor", xgb_preprocessor),
    ("regressor",    XGBRegressor(
        random_state=42, max_depth=3,
        colsample_bytree=0.5, reg_lambda=1, gamma=0.2,
    )),
])

tscv   = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(xgb_pipe, X, y, cv=tscv, scoring="neg_root_mean_squared_error")
fold_rmse = -scores

print("XGBoost TimeSeriesSplit CV RMSE per fold:")
for i, r in enumerate(fold_rmse, 1):
    print(f"  Fold {i}: {r:.4f}")
print(f"\nMean: {fold_rmse.mean():.4f}  |  Std: {fold_rmse.std():.4f}")


In [ ]:
# Final fit and test evaluation
xgb_pipe.fit(X_train, y_train)
xgb_preds_test  = xgb_pipe.predict(X_test)
xgb_preds_train = xgb_pipe.predict(X_train)

xgb_rmse_test = np.sqrt(mean_squared_error(y_test, xgb_preds_test))
xgb_r2_test   = r2_score(y_test, xgb_preds_test)
xgb_mape_test = mean_absolute_percentage_error(y_test, xgb_preds_test) * 100
xgb_rmse_train = np.sqrt(mean_squared_error(y_train, xgb_preds_train))
xgb_r2_train   = r2_score(y_train, xgb_preds_train)

print(f"XGBoost Train  RMSE: {xgb_rmse_train:.4f}  R²: {xgb_r2_train:.4f}")
print(f"XGBoost Test   RMSE: {xgb_rmse_test:.4f}  R²: {xgb_r2_test:.4f}  MAPE: {xgb_mape_test:.2f}%")
print(f"\nGeneralisation gap (Train RMSE - Test RMSE): {xgb_rmse_test - xgb_rmse_train:.4f}")


In [ ]:
# CV fold-by-fold trend
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([f"Fold {i+1}" for i in range(5)], fold_rmse, marker="o", lw=2.2, color="#4C72B0", ms=9)
axes[0].fill_between(range(5), fold_rmse, alpha=0.12, color="#4C72B0")
axes[0].axhline(fold_rmse.mean(), ls="--", color="#DD4444", lw=1.5, label=f"Mean: {fold_rmse.mean():.2f}")
axes[0].set_title("CV RMSE improves as training window grows — model needs history", fontsize=10)
axes[0].set_ylabel("RMSE (USD)"); axes[0].legend()
axes[0].set_xticks(range(5)); axes[0].set_xticklabels([f"Fold {i+1}" for i in range(5)])

axes[1].scatter(y_test, xgb_preds_test, alpha=0.45, s=22, color="#4C72B0", label="Test predictions")
lims = [min(y_test.min(), xgb_preds_test.min())-0.3, max(y_test.max(), xgb_preds_test.max())+0.3]
axes[1].plot(lims, lims, "r--", lw=1.8, label="Perfect prediction")
axes[1].set_xlabel("Actual (USD)"); axes[1].set_ylabel("Predicted (USD)")
axes[1].set_title(f"Test set: R²={xgb_r2_test:.4f}, RMSE=${xgb_rmse_test:.4f}", fontsize=10)
axes[1].legend()

plt.suptitle("XGBoost on held-out test data", y=1.02, fontsize=12)
plt.tight_layout()
plt.savefig(ROOT / "outputs/figures/07_actual_vs_predicted_xgb.png", dpi=150, bbox_inches="tight")
plt.show()


**The CV RMSE (mean 1.22) vs test RMSE (0.21) gap needs to be addressed directly.**

This looks alarming but it's explainable. The first CV fold trains on only ~160 rows and tests on ~160 — a very small training set that doesn't cover all the interaction patterns. As the folds grow (Fold 5 trains on 800+ rows), RMSE drops to 1.03. The test-set evaluation (800 train, 200 test) is the most informative number because it mirrors production conditions.

The deeper issue: `Previous_Price` does 53% of the work. It's a near-perfect proxy for price when training and test distributions are similar. In production, if you don't have reliable access to yesterday's price, the model degrades significantly. That's the honest limitation.
